In [ ]:
#Import Statements Here
import numpy as np
import pandas as pd
from statistics import mean
import seaborn as sns
import matplotlib.pyplot as plt
import sklearn as skl
from sklearn import metrics
from scipy import stats
rng = np.random.default_rng()
from sklearn import linear_model
from sklearn import metrics
from sklearn.datasets import make_classification
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import roc_curve
from sklearn.metrics import roc_auc_score
from matplotlib import pyplot
from sklearn.calibration import CalibratedClassifierCV
import shap
import os
import subprocess

from matplotlib import cm
import matplotlib as mpl
from matplotlib.colors import ListedColormap

In [ ]:
def getFromBucket(name_of_file_in_bucket):
    # get the bucket name
    my_bucket = os.getenv('WORKSPACE_BUCKET')

    # copy csv file from the bucket to the current working space
    os.system(f"gsutil cp '{my_bucket}/data/{name_of_file_in_bucket}' .")

    print(f'[INFO] {name_of_file_in_bucket} is successfully downloaded into your working space')
    # save dataframe in a csv file in the same workspace as the notebook
    my_dataframe = pd.read_csv(name_of_file_in_bucket)
    return my_dataframe.reset_index(drop=True)

In [ ]:
covid_features = getFromBucket('long_covid_features_2_25_2025.csv')

survey_pred_vars = list(covid_features.loc[covid_features['survey_feature']==1,'feature_name'])

gene_pred_vars = list(covid_features.loc[covid_features['genetic_feature']==1,'feature_name'])

ehr_pred_vars = list(covid_features.loc[covid_features['ehr_feature']==1,'feature_name'])

mobile_device_pred_vars = list(covid_features.loc[covid_features['mobile_device_feature']==1,'feature_name'])

all_pred_vars = list(survey_pred_vars) + list(gene_pred_vars) + list(ehr_pred_vars)
print(len(all_pred_vars))

column_list = all_pred_vars

In [ ]:
def download_from_bucket(name_of_file_in_bucket):
    # get the bucket name
    my_bucket = os.getenv('WORKSPACE_BUCKET')

    # copy csv file from the bucket to the current working space
    os.system(f"gsutil cp '{my_bucket}/data/{name_of_file_in_bucket}' .")

    print(f'[INFO] {name_of_file_in_bucket} is successfully downloaded into your working space')

In [ ]:
# This snippet assumes that you run setup first

# This code lists objects in your Google Bucket

# Get the bucket name
my_bucket = os.getenv('WORKSPACE_BUCKET')

# List objects in the bucket
print(subprocess.check_output(f"gsutil ls -r {my_bucket}", shell=True).decode('utf-8'))

In [ ]:
split_id = 0
split_path = os.path.join("splits", f"split_{split_id}")
download_from_bucket(os.path.join(split_path, "test.csv"))

In [ ]:
SPLIT_list = list(range(10))
FOLD_list = list(range(5))
output_prefix = 'xgboost_all_features'

feature_importance_value = {}
correlation_coeff_value = {}

shap_value_cum = []
test_temp_cum = []

for split_id in SPLIT_list:
    print('**************************  %d   **************************' % split_id)
    split_path = os.path.join("splits", f"split_{split_id}")
    download_from_bucket(os.path.join(split_path, "test.csv"))  ######## new line for Chris
    test_df = pd.read_csv(os.path.join(split_path, "test.csv"))
    #test_df = getFromBucket(os.path.join(split_path, "test.csv"))
    test_data = test_df.drop(columns=["target"])
    test_data = test_data[all_pred_vars]
    test_data_label = test_df["target"]
        
    for fold_id in FOLD_list:
        print('         **************************  %d   **************************' % fold_id)
        test_temp_cum.extend(test_data.to_numpy())
        download_from_bucket(f'{output_prefix}_shap_values_split_{split_id}_fold_{fold_id}.npy')  ######## new line for Chris
        shap_values = np.load(f'{output_prefix}_shap_values_split_{split_id}_fold_{fold_id}.npy', allow_pickle=True)
        shap_value_cum.extend(shap_values)
        shap_df = pd.DataFrame(shap_values, columns = column_list)
        shap_df_abs = abs(shap_df)
        feature_importance = shap_df_abs.mean(axis=0).sort_values(ascending=False)
        for key, value in feature_importance.items():
            if key in feature_importance_value.keys():
                feature_importance_value[key].append(value)
            else:
                feature_importance_value[key] = [value]
        print(len(shap_value_cum))
        print(len(test_temp_cum))
        
        df_v = test_data.copy().reset_index().drop('index',axis=1)
        
        for key_ in list(test_data.columns):
            if shap_df[key_].std() == 0 or test_data[key_].std() == 0:
                continue
            corr_value = np.corrcoef(shap_df[key_],test_data[key_])[1][0]
            if key_ in correlation_coeff_value.keys():
                correlation_coeff_value[key_].append(corr_value)
            else:
                correlation_coeff_value[key_] = [corr_value]

In [ ]:
# SPLIT_list = list(range(10))
# FOLD_list = list(range(5))
# output_prefix = 'xgboost_all_features'

# feature_importance_value = {}
# correlation_coeff_value = {}

# shap_value_cum = []
# test_temp_cum = []

# for split_id in SPLIT_list:
#     print('**************************  %d   **************************' % split_id)
#     split_path = os.path.join("splits", f"split_{split_id}")
#     test_df = pd.read_csv(os.path.join(split_path, "test.csv"))
#     test_data = test_df.drop(columns=["target"])
#     test_data = test_data[all_pred_vars]
#     test_data_label = test_df["target"]
        
#     for fold_id in FOLD_list:
#         print('         **************************  %d   **************************' % fold_id)
#         test_temp_cum.extend(test_data.to_numpy())
        
#         shap_values = np.load(f'{output_prefix}_shap_values_split_{split_id}_fold_{fold_id}.npy', allow_pickle=True)
#         shap_value_cum.extend(shap_values)
#         shap_df = pd.DataFrame(shap_values, columns = column_list)
#         shap_df_abs = abs(shap_df)
#         feature_importance = shap_df_abs.mean(axis=0).sort_values(ascending=False)
#         for key, value in feature_importance.items():
#             if key in feature_importance_value.keys():
#                 feature_importance_value[key].append(value)
#             else:
#                 feature_importance_value[key] = [value]
#         print(len(shap_value_cum))
#         print(len(test_temp_cum))
        
#         df_v = test_data.copy().reset_index().drop('index',axis=1)
        
#         for key_ in list(test_data.columns):
#             if shap_df[key_].std() == 0 or test_data[key_].std() == 0:
#                 continue
#             corr_value = np.corrcoef(shap_df[key_],test_data[key_])[1][0]
#             if key_ in correlation_coeff_value.keys():
#                 correlation_coeff_value[key_].append(corr_value)
#             else:
#                 correlation_coeff_value[key_] = [corr_value]

In [ ]:
feature_importance

In [ ]:
feature_importance_value_ = {}
for key, value in feature_importance_value.items():
    feature_importance_value_[key] = np.mean(value)
feature_importance_df = pd.DataFrame.from_dict(feature_importance_value_, orient='index').fillna(0)
feature_importance_df.reset_index(inplace=True)
feature_importance_df.columns = ['Variable','SHAP_abs']
feature_importance_df

In [ ]:
feature_name_mapping = {
    'op_post_visit_ratio': '[E]: Outpatient Post-Visit Ratio',
    'dyspnea': '[E]: Dyspnea',
    'Active Duty: Active Duty Serve Status': '[S]: Active Duty Serve Status',
    'Overall Health: Average Fatigue 7 Days': '[S]: Overall Health: Average Fatigue 7 Days',
    'apprx_age': '[E]: Age',
    'hospitalized': '[E]: Hospitalization',
    'headache': '[E]: Headache',
    'Overall Health: Emotional Problem 7 Days': '[S]: Overall Health: Emotional Problem 7 Days',
    'Overall Health: Medical Form Confidence': '[S]: Overall Health: Medical Form Confidence',
    'ip_post_visit_ratio': '[E]: Inpatient Post-Visit Ratio',
    'fatigue': '[E]: Fatigue',
    'chr19:4719431:G:A_A': '[G]: chr19:4719431:G:A_A',
    'Biological Sex At Birth: Sex At Birth': '[S]: Biological Sex At Birth: Sex At Birth',
    'Overall Health: Average Pain 7 Days': '[S]: Overall Health: Average Pain 7 Days',
    'Electronic Smoking: Electric Smoke Participant': '[S]: Electronic Smoking: Electric Smoke Participant',
    'chr10:79946568:A:G_G': '[G]: chr10:79946568:A:G_G',
    'Overall Health: Health Material Assistance': '[S]: Overall Health: Health Material Assistance',
    'palpitations': '[E]: Palpitations',
    'pain_of_truncal_structure': '[E]: Pain Of Truncal Structure',
    'mental_disorder': '[E]: Ment

In [ ]:
viridis = cm.get_cmap('viridis', 256)
top = cm.get_cmap('Oranges_r', 128)
bottom = cm.get_cmap('Blues', 128)
newcolors = np.vstack((top(np.linspace(0, 1, 128)), bottom(np.linspace(0, 1, 128))))
newcmp = ListedColormap(newcolors, name='OrangeBlue')


def ABS_SHAP(feature_importance_value, correlation_coeff_value, top, colors):

    correlation_coeff_value_ = {}
    for key, value in correlation_coeff_value.items():
        correlation_coeff_value_[key] = np.mean(value)
    corr_df = pd.DataFrame.from_dict(correlation_coeff_value_, orient='index').fillna(0)
    corr_df.reset_index(inplace=True)
    corr_df.columns  = ['Variable','Corr']
    color_assigned = []
    for corr in list(corr_df['Corr']):
        if not np.isnan(corr):
            color_assigned.append(colors[int((corr+1)/2 * len(colors))])
        else:
            color_assigned.append([1,1,1,1])
    corr_df['Sign'] = color_assigned

    feature_importance_value_ = {}
    feature_importance_value_std = {}
    for key, value in feature_importance_value.items():
        feature_importance_value_[key] = np.mean(value)
        feature_importance_value_std[key] = np.std(value)

    feature_importance_std_df = pd.DataFrame.from_dict(feature_importance_value_std, orient='index').fillna(0)
    feature_importance_std_df.reset_index(inplace=True)
    feature_importance_std_df.columns = ['Variable','SHAP_std']

    feature_importance_df = pd.DataFrame.from_dict(feature_importance_value_, orient='index').fillna(0)
    feature_importance_df.reset_index(inplace=True)
    feature_importance_df.columns = ['Variable','SHAP_abs']

    feature_importance_df = feature_importance_df.merge(feature_importance_std_df, left_on='Variable',right_on='Variable',how='inner')

    k2 = feature_importance_df.merge(corr_df,left_on = 'Variable',right_on='Variable',how='inner')
    k2 = k2.sort_values(by='SHAP_abs',ascending = False).head(top).iloc[::-1]
    print(list(k2['Variable']))
    
    if top == 20:
        k2['Variable']= k2['Variable'].apply(lambda x: feature_name_mapping.get(x, x))
    
    colorlist = k2['Sign']
    ax = k2.plot.barh(x='Variable',y='SHAP_abs',color = colorlist, figsize=(8,10),legend=False#, xerr='SHAP_std'
                      , edgecolor='black')
    ax.set_xlabel("Feature contribution [SHAP Values].")
    ax.set_ylabel("Factors")
    
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.spines['bottom'].set_visible(False)
    ax.spines['left'].set_visible(False)
    ax.xaxis.grid(color='gray', linestyle='-.', linewidth=0.5)
#     plt.savefig('feature_imp_0.png', dpi=300) 
    plt.savefig(f'feature_imp_0_top{top}.png', dpi=300, bbox_inches='tight')

    fig, aaxx = plt.subplots(figsize=(0.2, 15))
    fig.subplots_adjust(bottom=0.5)
    cmap = mpl.colors.ListedColormap(colors)
    norm = mpl.colors.Normalize(vmin=-1, vmax=1)
    cb1 = mpl.colorbar.ColorbarBase(aaxx, cmap=cmap,
                                norm=norm, orientation='vertical')
    plt.savefig(f'feature_imp_1_top{top}.png', dpi=300, bbox_inches='tight')
    
    return k2

In [ ]:
k20 = ABS_SHAP(feature_importance_value, correlation_coeff_value, 20, newcolors)

In [ ]:
pd.DataFrame(correlation_coeff_value)

In [ ]:
pd.DataFrame(feature_importance_value)

In [ ]:
feature_importance_value_ = {}
for key, value in correlation_coeff_value.items():
    correlation_coeff_value_[key] = np.mean(value)
cor_feature_importance_df = pd.DataFrame.from_dict(correlation_coeff_value_, orient='index').fillna(0)
cor_feature_importance_df.reset_index(inplace=True)
cor_feature_importance_df.columns = ['Variable','SHAP_corr']
#pd.concat([cor_feature_importance_df, feature_importance_df], axis=1)
pd.merge(feature_importance_df, cor_feature_importance_df, on='Variable',how='outer')
#cor_feature_importance_df

In [ ]:
pd.merge(feature_importance_df, cor_feature_importance_df, on='Variable',how='outer').to_csv('variable_feature_importance.csv')
correlation_coeff_value_ = {}
for key, value in correlation_coeff_value.items():
    correlation_coeff_value_[key] = np.mean(value)
corr_df = pd.DataFrame.from_dict(correlation_coeff_value_, orient='index').fillna(0)
corr_df.reset_index(inplace=True)
corr_df.columns  = ['Variable','Corr']
color_assigned = []
for corr in list(corr_df['Corr']):
    if not np.isnan(corr):
        color_assigned.append(colors[int((corr+1)/2 * len(colors))])
    else:
        color_assigned.append([1,1,1,1])
corr_df['Sign'] = color_assigned

feature_importance_value_ = {}
feature_importance_value_std = {}
for key, value in feature_importance_value.items():
    feature_importance_value_[key] = np.mean(value)
    feature_importance_value_std[key] = np.std(value)

feature_importance_std_df = pd.DataFrame.from_dict(feature_importance_value_std, orient='index').fillna(0)
feature_importance_std_df.reset_index(inplace=True)
feature_importance_std_df.columns = ['Variable','SHAP_std']

feature_importance_df = pd.DataFrame.from_dict(feature_importance_value_, orient='index').fillna(0)
feature_importance_df.reset_index(inplace=True)
feature_importance_df.columns = ['Variable','SHAP_abs']

feature_importance_df = feature_importance_df.merge(feature_importance_std_df, left_on='Variable',right_on='Variable',how='inner')

k2 = feature_importance_df.merge(corr_df,left_on = 'Variable',right_on='Variable',how='inner')
k2 = k2.sort_values(by='SHAP_abs',ascending = False).iloc[::-1]

k2['type'] = k2['Variable'].map(
    lambda x: 'Survey' if x in survey_pred_vars else 
              'Gene' if x in gene_pred_vars else 
              'EHR' if x in ehr_pred_vars else 
              'Other'
)

type_agg = type_agg = k2.groupby('type').agg({
    'SHAP_abs': 'sum',
    'Corr': 'mean'
}).reset_index()
type_agg = type_agg.sort_values('SHAP_abs', ascending=False)

In [ ]:
# 2. Assign colors based on correlation
norm = mpl.colors.Normalize(vmin=-1, vmax=1)
color_indices = ((type_agg['Corr'] + 1) / 2 * (len(colors) - 1)).astype(int)
color_list = [colors[i] for i in color_indices]

# Plot
fig, ax = plt.subplots(figsize=(6, 2))
ax.barh(
    y=type_agg['type'],
    width=type_agg['SHAP_abs'],
    color=color_list,
    edgecolor='black'
)

ax.set_xlabel("Total Feature Contribution [SHAP Values]")
ax.set_ylabel("Feature Type")
ax.set_title("SHAP by Feature Type (Color = Corr)")

ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.spines['bottom'].set_visible(False)
ax.spines['left'].set_visible(False)
ax.xaxis.grid(color='gray', linestyle='-.', linewidth=0.5)

plt.tight_layout()
plt.savefig("feature_type_SHAP_corr.png", dpi=300, bbox_inches='tight')
plt.show()

# Optional: Add vertical colorbar
fig, aaxx = plt.subplots(figsize=(0.2, 3))
fig.subplots_adjust(bottom=0.5)
cb1 = mpl.colorbar.ColorbarBase(
    aaxx, cmap=cmap, norm=norm, orientation='vertical'
)
plt.savefig("feature_type_SHAP_corr_colorbar.png", dpi=300, bbox_inches='tight')

In [ ]:
correlation_coeff_value_ = {}
for key, value in correlation_coeff_value.items():
    correlation_coeff_value_[key] = np.mean(value)
corr_df = pd.DataFrame.from_dict(correlation_coeff_value_, orient='index').fillna(0)
corr_df.reset_index(inplace=True)
corr_df.columns  = ['Variable','Corr']
color_assigned = []
for corr in list(corr_df['Corr']):
    if not np.isnan(corr):
        color_assigned.append(colors[int((corr+1)/2 * len(colors))])
    else:
        color_assigned.append([1,1,1,1])
corr_df['Sign'] = color_assigned

feature_importance_value_ = {}
feature_importance_value_std = {}
for key, value in feature_importance_value.items():
    feature_importance_value_[key] = np.mean(value)
    feature_importance_value_std[key] = np.std(value)

feature_importance_std_df = pd.DataFrame.from_dict(feature_importance_value_std, orient='index').fillna(0)
feature_importance_std_df.reset_index(inplace=True)
feature_importance_std_df.columns = ['Variable','SHAP_std']

feature_importance_df = pd.DataFrame.from_dict(feature_importance_value_, orient='index').fillna(0)
feature_importance_df.reset_index(inplace=True)
feature_importance_df.columns = ['Variable','SHAP_abs']

feature_importance_df = feature_importance_df.merge(feature_importance_std_df, left_on='Variable',right_on='Variable',how='inner')

k2 = feature_importance_df.merge(corr_df,left_on = 'Variable',right_on='Variable',how='inner')
k2 = k2.sort_values(by='SHAP_abs',ascending = False).iloc[::-1]

k2['type'] = k2['Variable'].map(
    lambda x: 'Survey' if x in survey_pred_vars else 
              'Gene' if x in gene_pred_vars else 
              'EHR' if x in ehr_pred_vars else 
              'Other'
)

type_agg = type_agg = k2.groupby('type').agg({
    'SHAP_abs': 'mean',
    'Corr': 'mean'
}).reset_index()
type_agg = type_agg.sort_values('SHAP_abs', ascending=False)

# 2. Assign colors based on correlation
norm = mpl.colors.Normalize(vmin=-1, vmax=1)
color_indices = ((type_agg['Corr'] + 1) / 2 * (len(colors) - 1)).astype(int)
color_list = [colors[i] for i in color_indices]

# Plot
fig, ax = plt.subplots(figsize=(6, 2))
ax.barh(
    y=type_agg['type'],
    width=type_agg['SHAP_abs'],
    color=color_list,
    edgecolor='black'
)

ax.set_xlabel("Average Feature Contribution [SHAP Values]")
ax.set_ylabel("Feature Type")
ax.set_title("SHAP by Feature Type (Color = Corr)")

ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.spines['bottom'].set_visible(False)
ax.spines['left'].set_visible(False)
ax.xaxis.grid(color='gray', linestyle='-.', linewidth=0.5)

plt.tight_layout()
plt.savefig("avg_feature_type_SHAP_corr.png", dpi=300, bbox_inches='tight')
plt.show()

# Optional: Add vertical colorbar
fig, aaxx = plt.subplots(figsize=(0.2, 3))
fig.subplots_adjust(bottom=0.5)
cb1 = mpl.colorbar.ColorbarBase(
    aaxx, cmap=cmap, norm=norm, orientation='vertical'
)
plt.savefig("avg_feature_type_SHAP_corr_colorbar.png", dpi=300, bbox_inches='tight')